# Hour 3 · Letters as social networks


**Goal:** turn the corpus of **letters** into a graph of correspondents.

**Needs:** `pip install networkx`.

Ugaritic letters open with a fixed address formula: **`tḥm X`** = “message of X” (sender) and **`l Y rgm`** = “to Y, say” (recipient). We parse that.

*Reading:* `docs/06-letters-networks.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first ===
# Loads the Copenhagen Ugaritic Corpus (CUC) from the HuggingFace JSONL cache. No edits needed.
import sys
sys.path.append("..")                       # so Python can find data/loader.py
from data.loader import load_texts

texts = load_texts()                        # 278 real KTU tablets; cached after first download
print(f"Loaded {len(texts)} tablets.")
texts[0]["ktu"], texts[0]["genre"], texts[0]["lines"][:2]

## 1. Parse (sender → recipient) from the address formula

In [ ]:
# particles that are not names (drop them if the formula parse grabs one)
STOP = {"l", "w", "b", "k", "ʿm", "il", "ilm"}

def parse_letter(t):
    toks = t["tokens"]
    sender = recipient = None
    for i, w in enumerate(toks[:-1]):
        if w == "tḥm" and sender is None:
            sender = toks[i+1]
        if w == "l" and recipient is None:
            recipient = toks[i+1]
    return sender, recipient

edges = []
for t in texts:
    if t["genre"] != "letter": continue
    s, r = parse_letter(t)
    if s and r and s != r and s not in STOP and r not in STOP:
        edges.append((s, r, t["ktu"]))
print(len(edges), "edges parsed")
edges[:12]

## 2. Build the directed correspondence graph

In [ ]:
import networkx as nx
G = nx.DiGraph()
for s, r, k in edges:
    G.add_edge(s, r, ktu=k)
print("people:", G.number_of_nodes(), " letters:", G.number_of_edges())

## 3. Draw it

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(11,8))
pos = nx.spring_layout(G, seed=1, k=0.6)
sizes = [300 + 400*G.degree(n) for n in G.nodes()]
nx.draw(G, pos, with_labels=True, node_size=sizes, node_color="#cfe8ff",
        font_size=8, arrows=True, edge_color="#999")
plt.title("Ugaritic correspondence network (CUC letters)"); plt.show()

## 4. Who is central?

In [ ]:
import pandas as pd
deg = nx.degree_centrality(G)
pd.Series(deg).sort_values(ascending=False).head(10)

## 5. Discussion
- Central nodes are often titles/relations (*mlk* king, *umy* “my mother/lady”, *adtny* “our lady”) rather than unique persons — a real interpretive caveat.
- **Fragmentary preservation** biases the graph; a missing edge ≠ no relationship.

> **Production version:** UgaritLab `dulat/scripts/build_name_graph.py` builds a co-occurrence graph of **named entities** (PN/DN/TN/RN) pulled from the DULAT dictionary and matched in UDB tablets — far richer than the formula parser above.